# Promoter sequence analysis in Python

A portfolio edition of the completed analysis in **5BBB0238 Summative Coursework 2025–6**, Introduction to Programming for Bioscientists.

This notebook searches course-provided DNA sequences for exact motifs and reports candidate transcription start sites using the coursework spacing rule. These are sequence matches, not experimentally validated sites.

The source code has been organised with AI assistance. The original coursework file is unchanged. The 100-base context display has been corrected and file paths updated; see `../EDITING_NOTES.md`. The unfinished sequence-generation exercise is excluded.

All reported positions use **zero-based indexing**. Run the cells from top to bottom.

In [1]:
from pathlib import Path

# Work from either the repository root or its notebooks directory.
ROOT = Path.cwd()
if not (ROOT / "data" / "seq1.txt").is_file():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
assert (DATA_DIR / "seq1.txt").is_file(), "Open this notebook from the repository."


## 1. Check for a TATA-box motif

Read `seq1.txt` and check whether it contains the exact string `TATAA`.

In [2]:
file = open(DATA_DIR / "seq1.txt","r")
seq1 = file.read().strip()
file.close()

if "TATAA" in seq1:
    print("The motif'TATAA' is present in the sequence")
else:
    print("The motif'TATAA' is not present in the sequence")

The motif'TATAA' is present in the sequence


## 2. Locate the first match

Scan the sequence with a five-base window. Report the first matching index.

In [3]:
file = open(DATA_DIR / "seq1.txt","r")
seq1 = file.read().strip()
file.close()
motif_or_not = 0
for i in range(len(seq1) - 5 + 1):
    if seq1[i:i+5] == 'TATAA':
        tata_pos = i
        motif_or_not += 1
        break
if motif_or_not == 1:
    print("The motif is found")
    print(tata_pos)
else:
    print("The motif is not found")

The motif is found
432


## 3. Search for initiator-like dinucleotides

Within the first 30 bases, find adjacent pairs with C or T followed by A or G. Both bases must fall within this region.

In [4]:
file = open(DATA_DIR / "seq1.txt","r")
seq1 = file.read().strip()
file.close()
Y =['C', 'T']
R =['A', 'G']
inr_list = []
for i in range(29):
    if seq1[i] in Y and seq1[i+1] in R:
        inr_list.append(i)
print(inr_list)
print(len(inr_list))
for i in inr_list:
    print(seq1[i:i+2])

[0, 6, 8, 14, 23]
5
CA
TG
TG
CA
TG


## 4. Find candidate transcription start sites

In `seq2.txt`, pair each `TATAA` match with initiator-like dinucleotides. Retain pairs where the second base of the dinucleotide is 24–31 bases downstream of the start of `TATAA`. Report that second base as the candidate TSS.

In [5]:
file2 = open(DATA_DIR / 'seq2.txt','r')
seq2 = file2.read().strip()
file2.close()
tata_pos = []
for i in range(len(seq2) - 5 + 1):
    if seq2[i:i+5] == 'TATAA':
        tata_pos.append(i)
Y =['C', 'T']
R =['A', 'G']
inr_list = []
for i in range(len(seq2)-1):
    if seq2[i] in Y and seq2[i+1] in R:
        inr_list.append(i)

TSS_list = []
for i in tata_pos:
    for j in inr_list:
        second_nu = j + 1
        if 24 <= second_nu - i <= 31:
            TSS_list.append(second_nu)
print(TSS_list)

[854, 2259, 3240, 3243]


## 5. Package the search in a function

Repeat the search using a filename argument. Display 50 bases before each candidate TSS and 50 bases starting at it, separated by a space. Near a sequence boundary the displayed context is shorter.

**Portfolio correction:** the original cell displayed only the 100 bases upstream. The corrected window includes the TSS and downstream sequence; the candidate-search logic is unchanged.

In [6]:
def find_promoters(filename):
    file = open(filename,'r')
    seq = file.read().strip()
    file.close()
    tata_pos = []
    for i in range(len(seq) - 5 + 1):
        if seq[i:i+5] == 'TATAA':
            tata_pos.append(i)
    Y =['C', 'T']
    R =['A', 'G']
    inr_list = []
    for i in range(len(seq)-1):
        if seq[i] in Y and seq[i+1] in R:
            inr_list.append(i)

    TSS_list = []
    for i in tata_pos:
        for j in inr_list:
            second_nu = j + 1
            if 24 <= second_nu - i <= 31:
                TSS_list.append(second_nu)
    for i in TSS_list:
        # Corrected context: 50 bases before and 50 bases from the TSS.
        print(seq[max(0, i-50):i], seq[i:min(len(seq), i+50)])
    return TSS_list
find_promoters(DATA_DIR / "seq2.txt")

ATGACTTCAACAAATTTACCTATAAAGCCTCCTGAAAATTAAGGAGTTTC AAATGTCAGAAGAAGGAAATTCTGAGTTGATATTGAATCAAGCTTTACTC
AAGGCCGGGCACTGTGGCTCATGCCTATAATCTCAGCACTTTGGGAGGCC GAGGAGGGTGGATCACAAGGTCAGGAGTTTGAGACCAGCCTGGCCAATAT
TTCCACAGAAACACAAACAGAGATTGTATAAAGCACAAGGCATTATTCAT GCTACCCTTTGTCCTCCAGTCCAGTGGAGGAGAAAAGGAACAATCCCTGG
CACAGAAACACAAACAGAGATTGTATAAAGCACAAGGCATTATTCATGCT ACCCTTTGTCCTCCAGTCCAGTGGAGGAGAAAAGGAACAATCCCTGGCCA


[854, 2259, 3240, 3243]

## 6. Define the course motif dictionary

Use the five simplified binding-site strings supplied for this exercise. Exact matches to these short strings do not demonstrate protein binding.

In [7]:
tfbs = {"PDX1":	"TCTAAT",
"NKX6-1": "TAATTAA",
"MAFA":"TGCA",
"SP1":"CCGCCC",
"NFY":"CCAAT"}

## 7. Locate exact motif matches

Search `seq1.txt` for all matches, including overlapping matches, and report their zero-based start positions.

In [8]:
file = open(DATA_DIR / "seq1.txt","r")
seq1 = file.read().strip()
file.close()
for TF in tfbs:
    for i in range(len(seq1) - len(tfbs[TF]) + 1):
        if seq1[i:i+len(tfbs[TF])] == tfbs[TF]:
            print(TF, i)

PDX1 143
PDX1 246
NKX6-1 392
MAFA 122
MAFA 334
MAFA 479
SP1 116


## Interpretation and limits

The analysis reports four candidate TSS positions under the coursework rule. It searches only the supplied forward sequence; it does not evaluate the reverse complement, motif scores, expression levels or experimental binding.

The source notebook also contains an unfinished random sequence-generation exercise. It is outside this completed analysis and has not been presented as a result.